# HF-native Mistral-7B inference test (Blackwell sm_120 workarounds)

**Goal:** figure out whether we can run `mistralai/Mistral-7B-Instruct-v0.3` from Hugging Face directly (not via GGUF / llama.cpp) on this RTX 5090 so Day 5 A/B can compare **real base Mistral vs NyayaGPT INT4 GGUF**.

**Known blocker:** `cuBLAS` 12.8 has a bug on sm_120 where `cublasGemmEx(..., CUDA_R_16F/CUDA_R_16BF, ..., CUBLAS_GEMM_DEFAULT_TENSOR_OP)` fails with `CUBLAS_STATUS_INVALID_VALUE` during Mistral decode. Documented in `document.md`.

**Four attempts in order of promise:**
| # | Approach | Hypothesis |
|---|---|---|
| 1 | **BnB 4-bit nf4** | `bitsandbytes` uses its own CUDA kernels (gemv_4bit) — bypasses cuBLAS 16-bit gemm entirely |
| 2 | **FP32 full precision** | FP32 gemm uses `CUDA_R_32F` kernels, different code path from the broken 16-bit one |
| 3 | **BnB 4-bit + eager attn** | Alternative if default attention fails |
| 4 | **FP16 default (control)** | Confirms the bug still reproduces — negative control |

**CRITICAL:** run this in a **fresh kernel** that has never imported Unsloth. Unsloth globally patches `MistralAttention` on import, which breaks `attn_implementation="eager"` in the raw `transformers` path.

In [8]:
# Preflight — verify fresh kernel, no Unsloth pollution
import sys
assert 'unsloth' not in sys.modules, (
    'Unsloth is already imported in this kernel. Restart the kernel before running this notebook.'
)
assert 'FastLanguageModel' not in dir(), 'FastLanguageModel already in scope — restart kernel.'

import torch, transformers, bitsandbytes, gc, time
print(f'torch        : {torch.__version__}')
print(f'transformers : {transformers.__version__}')
print(f'bitsandbytes : {bitsandbytes.__version__}')
print(f'CUDA         : {torch.version.cuda}')
print(f'Device       : {torch.cuda.get_device_name()}')
print(f'Capability   : {torch.cuda.get_device_capability()}')
print(f'Free VRAM    : {torch.cuda.mem_get_info()[0] / 1e9:.1f} GB / {torch.cuda.mem_get_info()[1] / 1e9:.1f} GB')

torch        : 2.10.0+cu128
transformers : 4.56.2
bitsandbytes : 0.49.2
CUDA         : 12.8
Device       : NVIDIA GeForce RTX 5090
Capability   : (12, 0)
Free VRAM    : 25.2 GB / 34.2 GB


In [2]:
# Shared test harness — loads a model, runs one short generation, returns a result record
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from dataclasses import dataclass, field
from typing import Optional

MODEL_ID = 'mistralai/Mistral-7B-Instruct-v0.3'
TEST_QUESTION = 'What is the maximum punishment under Section 302 IPC?'
MAX_NEW_TOKENS = 40

@dataclass
class AttemptResult:
    name: str
    success: bool
    error: Optional[str] = None
    output: Optional[str] = None
    latency_ms: Optional[float] = None
    vram_used_gb: Optional[float] = None

def _free_vram():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()

def run_attempt(name: str, load_fn) -> AttemptResult:
    print(f'\n{"="*60}\nAttempt: {name}\n{"="*60}')
    _free_vram()
    try:
        tok = AutoTokenizer.from_pretrained(MODEL_ID)
        model = load_fn()
        model.eval()
        messages = [{'role': 'user', 'content': TEST_QUESTION}]
        prompt = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        ids = tok(prompt, return_tensors='pt').to(model.device)
        t0 = time.perf_counter()
        with torch.no_grad():
            out = model.generate(**ids, max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
                                 pad_token_id=tok.eos_token_id)
        torch.cuda.synchronize()
        elapsed_ms = (time.perf_counter() - t0) * 1000
        text = tok.decode(out[0][ids.input_ids.shape[1]:], skip_special_tokens=True).strip()
        vram_gb = torch.cuda.max_memory_allocated() / 1e9
        print(f'  ✓ SUCCESS in {elapsed_ms:.0f} ms, peak VRAM {vram_gb:.2f} GB')
        print(f'  Output: {text[:200]}')
        result = AttemptResult(name, True, output=text, latency_ms=elapsed_ms, vram_used_gb=vram_gb)
    except Exception as exc:
        err = f'{type(exc).__name__}: {str(exc)[:300]}'
        print(f'  ✗ FAILED — {err}')
        result = AttemptResult(name, False, error=err)
    finally:
        try:
            del model
        except NameError:
            pass
        _free_vram()
    return result

results = []
print('Test harness ready.')

Test harness ready.


## Attempt 1 — BnB 4-bit nf4

Uses `bitsandbytes` 4-bit quantization. bnb's `gemv_4bit_inference` kernel is a custom CUDA implementation that doesn't go through the broken cuBLAS 16-bit gemm path. ~5 GB VRAM.

In [3]:
def _load_bnb_4bit():
    bnb_cfg = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type='nf4',
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
    )
    return AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        quantization_config=bnb_cfg,
        device_map='auto',
    )

results.append(run_attempt('BnB 4-bit nf4 (compute=bf16)', _load_bnb_4bit))


Attempt: BnB 4-bit nf4 (compute=bf16)


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

  ✗ FAILED — RuntimeError: CUDA error: CUBLAS_STATUS_INVALID_VALUE when calling `cublasGemmEx( handle, opa, opb, m, n, k, &falpha, a, CUDA_R_16BF, lda, b, CUDA_R_16BF, ldb, &fbeta, c, std::is_same_v<C_Dtype, float> ? CUDA_R_32F : CUDA_R_16BF, ldc, compute_type, CUBLAS_GEMM_DEFAULT_TENSOR_OP)`


## Attempt 2 — FP32 full precision

`torch_dtype=torch.float32` forces FP32 matmul which uses `CUBLAS_R_32F` kernels — different code path from the broken 16-bit tensor-core path. ~28 GB VRAM, fits in 32 GB.

In [4]:
def _load_fp32():
    return AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        torch_dtype=torch.float32,
        device_map='auto',
        low_cpu_mem_usage=True,
    )

results.append(run_attempt('FP32', _load_fp32))


Attempt: FP32


`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

  ✗ FAILED — RuntimeError: CUDA error: CUBLAS_STATUS_INVALID_VALUE when calling `cublasSgemmStridedBatched( handle, opa, opb, m, n, k, &alpha, a, lda, stridea, b, ldb, strideb, &beta, c, ldc, stridec, num_batches)`


## Attempt 3 — BnB 4-bit with explicit eager attention

If attempt 1 failed because of an attention kernel issue, this forces the reference eager attention path. Only safe because this kernel has never imported Unsloth, so `MistralAttention` is not patched.

In [5]:
def _load_bnb_4bit_eager():
    bnb_cfg = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type='nf4',
        bnb_4bit_compute_dtype=torch.bfloat16,
    )
    return AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        quantization_config=bnb_cfg,
        device_map='auto',
        attn_implementation='eager',
    )

results.append(run_attempt('BnB 4-bit + eager attention', _load_bnb_4bit_eager))


Attempt: BnB 4-bit + eager attention


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

  ✗ FAILED — RuntimeError: CUDA error: CUBLAS_STATUS_INVALID_VALUE when calling `cublasGemmEx( handle, opa, opb, m, n, k, &falpha, a, CUDA_R_16BF, lda, b, CUDA_R_16BF, ldb, &fbeta, c, std::is_same_v<C_Dtype, float> ? CUDA_R_32F : CUDA_R_16BF, ldc, compute_type, CUBLAS_GEMM_DEFAULT_TENSOR_OP)`


## Attempt 4 — FP16 default (negative control)

**Expected to fail** with `CUBLAS_STATUS_INVALID_VALUE`. Confirms the bug still reproduces and our test is meaningful.

In [6]:
def _load_fp16():
    return AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        torch_dtype=torch.float16,
        device_map='auto',
    )

results.append(run_attempt('FP16 default (negative control)', _load_fp16))


Attempt: FP16 default (negative control)


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

  ✗ FAILED — RuntimeError: CUDA error: CUBLAS_STATUS_INVALID_VALUE when calling `cublasGemmEx( handle, opa, opb, m, n, k, alpha_ptr, a, CUDA_R_16F, lda, b, CUDA_R_16F, ldb, beta_ptr, c, std::is_same_v<C_Dtype, float> ? CUDA_R_32F : CUDA_R_16F, ldc, compute_type, CUBLAS_GEMM_DEFAULT_TENSOR_OP)`


## Summary

In [7]:
print(f'\n{"="*78}\n{"SUMMARY":^78}\n{"="*78}')
print(f'{"Approach":<38} {"Status":<8} {"Latency":>10} {"VRAM":>8}')
print('-' * 78)
for r in results:
    status = '✓ OK' if r.success else '✗ FAIL'
    lat    = f'{r.latency_ms:.0f} ms' if r.latency_ms is not None else '—'
    vram   = f'{r.vram_used_gb:.2f} GB' if r.vram_used_gb is not None else '—'
    print(f'{r.name:<38} {status:<8} {lat:>10} {vram:>8}')
    if not r.success and r.error:
        print(f'    ↳ {r.error[:120]}')
print('=' * 78)

working = [r for r in results if r.success and 'control' not in r.name]
if working:
    fastest = min(working, key=lambda r: r.latency_ms)
    print(f'\n✓ Recommended HF-native path for Day 5: {fastest.name}')
    print(f'  Latency: {fastest.latency_ms:.0f} ms for {MAX_NEW_TOKENS} tokens')
    print(f'  VRAM:    {fastest.vram_used_gb:.2f} GB')
else:
    print('\n✗ No working HF-native path found. Stick with GGUF base Mistral for Day 5.')


                                   SUMMARY                                    
Approach                               Status      Latency     VRAM
------------------------------------------------------------------------------
BnB 4-bit nf4 (compute=bf16)           ✗ FAIL            —        —
    ↳ RuntimeError: CUDA error: CUBLAS_STATUS_INVALID_VALUE when calling `cublasGemmEx( handle, opa, opb, m, n, k, &falpha, a,
FP32                                   ✗ FAIL            —        —
    ↳ RuntimeError: CUDA error: CUBLAS_STATUS_INVALID_VALUE when calling `cublasSgemmStridedBatched( handle, opa, opb, m, n, k
BnB 4-bit + eager attention            ✗ FAIL            —        —
    ↳ RuntimeError: CUDA error: CUBLAS_STATUS_INVALID_VALUE when calling `cublasGemmEx( handle, opa, opb, m, n, k, &falpha, a,
FP16 default (negative control)        ✗ FAIL            —        —
    ↳ RuntimeError: CUDA error: CUBLAS_STATUS_INVALID_VALUE when calling `cublasGemmEx( handle, opa, opb, m, n, k, alpha